# OpenPlaque — LAD Distal Bidirectional Validation v1
Independent reverse-trace validation of the source-supported 15.2-mm distal LAD candidate. Frozen anatomy is not modified. Use **Runtime → Run all**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, sys
os.chdir('/content')
DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
OUTPUT = DRIVE_ROOT / 'LAD_Distal_Bidirectional_Validation_v1'
REUSE_EXISTING_OUTPUT = False
if OUTPUT.exists() and not REUSE_EXISTING_OUTPUT:
    shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / 'notebook_started.json').write_text(json.dumps({'status':'STARTED','notebook':'LAD_Distal_Bidirectional_Validation_v1'}, indent=2))
print('Output:', OUTPUT)

In [ ]:
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
BRANCH = 'lad-distal-bidirectional-consensus-from-main'
PINNED_SCIENCE_COMMIT = 'bb57eade2a6f0fd61bfcca0762bf03b6092b02ce'
repo = '/content/OpenPlaque_lad_distal_bidirectional_v1'
os.chdir('/content')
if os.path.exists(repo):
    shutil.rmtree(repo)
!git clone --depth 20 --branch "$BRANCH" https://github.com/pazzani/OpenPlaque.git "$repo"
!git -C "$repo" checkout --detach "$PINNED_SCIENCE_COMMIT"
HEAD_OUT = get_ipython().getoutput(f'git -C {repo} rev-parse HEAD')
MB_OUT = get_ipython().getoutput(f'git -C {repo} merge-base HEAD {BASELINE}')
HEAD = HEAD_OUT[-1].strip() if HEAD_OUT else ''
MB = MB_OUT[-1].strip() if MB_OUT else ''
print('Checked out:', HEAD)
print('Merge base:', MB)
assert HEAD == PINNED_SCIENCE_COMMIT, HEAD_OUT
assert MB == BASELINE, MB_OUT
%pip install -q /content/OpenPlaque_lad_distal_bidirectional_v1
for name in list(sys.modules):
    if name == 'openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
os.chdir(repo)
print('Repository cwd:', os.getcwd())

In [ ]:
import pytest
from openplaque.lad_distal_bidirectional_validation_v1 import synthetic_self_test
print('Synthetic self-test:', synthetic_self_test())
rc = pytest.main(['-q', 'tests/test_lad_distal_bidirectional_validation_v1.py'])
assert rc == 0, f'pytest failed with code {rc}'

In [ ]:
prior = DRIVE_ROOT / 'LAD_Distal_Endpoint_Continuation_v1'
required = [
    DRIVE_ROOT / 'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
    DRIVE_ROOT / 'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
    DRIVE_ROOT / 'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
    prior / 'summary.json',
    prior / 'best_independent_distal_extension.csv',
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, f'Missing prerequisites: {missing}'
prior_summary = json.loads((prior / 'summary.json').read_text())
print('Prior status:', prior_summary.get('status'))
print('Prior forward length:', prior_summary.get('best_target', {}).get('accepted_arc_mm'))
preflight = {'status':'COMPLETE','science_commit':PINNED_SCIENCE_COMMIT,'prior_status':prior_summary.get('status'),'required_files_present':True}
(OUTPUT / 'preflight_complete.json').write_text(json.dumps(preflight, indent=2))

In [ ]:
from openplaque.lad_distal_bidirectional_validation_v1 import run
result = run(drive_root=str(DRIVE_ROOT), output_dir=str(OUTPUT))
print(json.dumps(result['summary'], indent=2, default=str))
print('Report:', result.get('report'))
print('ZIP:', result.get('zip'))